In [ ]:
# SECTION 0 — IMPORTS
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocessing
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# Model selection
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb        # pip install xgboost
import lightgbm as lgb        # pip install lightgbm


# ── Scikit-learn: evaluation metrics ───
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score,
    auc,
)
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight

import warnings
warnings.filterwarnings('ignore')   # suppress convergence and deprecation noise

# ── Global plot style ──

sns.set_theme(style='whitegrid', font_scale=1.0)

# Colour palette: 7 distinct colours — one per bean variety
PALETTE = {
    'BARBUNYA': '#E07B54',
    'BOMBAY'  : '#5B8DB8',
    'CALI'    : '#6CB87A',
    'DERMASON': '#B87BB8',
    'HOROZ'   : '#D4A83A',
    'SEKER'   : '#5BA8B8',
    'SIRA'    : '#B85B5B',
}

RANDOM_STATE = 42   # fix all random seeds for reproducibility
FIGSIZE      = (14, 5)


In [ ]:

# SECTION 1 — DATA LOADING & FIRST INSPECTION
# ─────────────────────────────────────────────────────────────────────────────


print("=" * 65)
print("  SECTION 1 — DATA LOADING & INSPECTION")
print("=" * 65)

df = pd.read_excel('Beans_Dataset.xlsx')   # ← adjust path if needed

print(f"\n  Dataset shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\n  Column names:\n  {df.columns.tolist()}")

# .head() previews the first 5 rows — quick sanity check on column types
print(f"\n  First 5 rows:\n{df.head()}")

# .info() shows dtype and non-null count for every column
print("\n  Column info:")
df.info()

# .describe() gives count, mean, std, min, quartiles, max for numeric columns
print("\n  Descriptive statistics:")
print(df.describe().round(3).T)   # .T transposes for easier reading

In [ ]:

# SECTION 2 — DATA QUALITY CHECKS
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 65)
print("  SECTION 2 — DATA QUALITY CHECKS")
print("=" * 65)

# (a) Missing values per column
missing = df.isnull().sum()
pct     = (missing / len(df) * 100).round(3)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': pct})
print("\n Missing values:")
print(missing_df[missing_df['Missing Count'] > 0])
# Result: 6 NaN cells across MajorAxisLength, AspectRation,
#         ConvexArea, Solidity, Compactness — negligible (< 0.05%)

# (b) Duplicate rows
n_dup = df.duplicated().sum()
print(f"\n  Duplicate rows: {n_dup}")

# (c) Class distribution — visualise
print("\n  Class distribution:")
class_counts = df['Class'].value_counts()
print(class_counts)

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

# Bar chart of raw counts
sns.barplot(x=class_counts.index, y=class_counts.values,
            palette=list(PALETTE.values()), ax=axes[0])
axes[0].set_title('Bean Class Distribution (absolute counts)')
axes[0].set_xlabel('Bean Variety')
axes[0].set_ylabel('Count')
for bar, val in zip(axes[0].patches, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{val:,}', ha='center', va='bottom', fontsize=9)

# Pie chart of proportions
axes[1].pie(class_counts.values, labels=class_counts.index,
            autopct='%1.1f%%', colors=list(PALETTE.values()),
            startangle=90, pctdistance=0.82)
axes[1].set_title('Class Proportions')

plt.suptitle('SECTION 2 — Class Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('sec2_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
# Observation: BOMBAY (522) is severely under-represented vs DERMASON (3,546).
# This imbalance means accuracy alone is a misleading metric — we use Macro-F1.

: 